In [ ]:
import polars as pl
from datetime import date, timedelta
from sklearn.metrics import classification_report

from fleetsense.features.data_loader import get_dataset, FEATURES
from fleetsense.model.base_model import (
    load_baseline_model,
    predict_baseline_proba,
)
from fleetsense.config import DATA_DATASET

from fleetsense.monitoring.distribution_monitoring import (
    load_baselines,
)
import matplotlib.pyplot as plt
import sys

sys.path.append("..")

from scripts.train_model import train, load_permutation_importance
import scripts.monitor as monitor

# ## 2. Train baseline model and build PSI reference on a fixed training window

In [ ]:
FULL_DATA_PATH = DATA_DATASET / "vessel_weekly_features.csv"
SAMPLE_DATA_PATH = DATA_DATASET / "vessel_weekly_features_sample.csv"
TRAIN_START = date(2025, 6, 1)  # your known baseline window, same as production training
TRAIN_END = date(2025, 8, 31)
train(start=TRAIN_START, end=TRAIN_END, data_path=FULL_DATA_PATH)  # actually calls training pipeline

model = load_baseline_model()  # the actual saved artifact
baselines = load_baselines()  # the actual saved baseline
importance_df = load_permutation_importance()
weights = importance_df["drift_weight"].to_dict()

In [ ]:
df = get_dataset(SAMPLE_DATA_PATH)  # load the dataset for evaluation
df = df.with_columns(pl.col("timestamp").str.to_datetime("%Y-%m-%dT%H:%M:%S%.f"))

i = 0
reports = {}
while True:
    starttime = TRAIN_END + timedelta(days=7 * i)
    endtime = starttime + timedelta(days=7)
    print(f"Evaluating week {i + 1}: {starttime} to {endtime}")
    test_df = df.filter(pl.col("timestamp").is_between(starttime, endtime, closed="left"))
    i += 1
    if len(test_df) == 0:
        break

    y_true = []
    y_pred = []
    for row in test_df.iter_rows(named=True):
        features = {k: row[k] for k in FEATURES}
        result = predict_baseline_proba(model, features)  # logs this prediction, as intended
        y_pred.append(result["vessel_type"])
        y_true.append(row["Ship type"])  # confirm this matches the dataset's actual column name

    report = classification_report(y_true, y_pred, output_dict=True)
    reports[starttime.strftime("%Y-%m")] = report

In [ ]:
classes = ["macro avg", "weighted avg"]
months = list(reports.keys())

fig, ax = plt.subplots(figsize=(10, 5))
for cls in classes:
    f1_scores = []
    for month in months:
        report = reports[month]
        f1 = report.get(cls, {}).get("f1-score", None)
        f1_scores.append(f1)
    ax.plot(months, f1_scores, marker="o", label=cls)

ax.set_xlabel("Month")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 Score over time (temporal drift)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
monitor.main()